# 05 - Feature engineering

**Job:** build the EDA-selected features, fit every transformation on
training data only, and apply the fitted objects to validation/test.

**Inputs:** split artifacts and `04_feature_decisions.json`.

**Outputs:** sparse feature matrices, labels/order IDs, fitted
preprocessor, feature list, and matrix metadata.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"
decisions_path = ARTIFACT_DIR / "04_feature_decisions.json"
assert decisions_path.exists(), "Run Notebook 04 first."
decisions = json.loads(decisions_path.read_text(encoding="utf-8"))

split_frames = {
    name: pd.read_csv(ARTIFACT_DIR / f"03_{name}.csv.gz", low_memory=False)
    for name in ["train", "validation", "test"]
}
for frame in split_frames.values():
    for column in ["order_purchase_timestamp", "order_estimated_delivery_date"]:
        frame[column] = pd.to_datetime(frame[column], errors="coerce")
print({name: frame.shape for name, frame in split_frames.items()})

{'train': (67529, 44), 'validation': (14470, 44), 'test': (14471, 44)}


## Build prediction-time features

The purchase timestamp and promised delivery date are known when the
order is placed. Their useful parts are derived below. Actual approval,
carrier hand-off, delivery, review, delay, status, target, and IDs are
never included in `X`.

In [2]:
fixed_holiday_month_days = {
    (1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)
}

def build_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame(index=frame.index)
    for column in decisions["base_numeric_features"]:
        features[column] = pd.to_numeric(frame[column], errors="coerce")
    for column in decisions["base_categorical_features"]:
        # scikit-learn expects np.nan rather than pandas.NA in
        # object-valued categorical inputs.
        features[column] = frame[column].astype(object).where(
            frame[column].notna(), np.nan
        )

    purchase = frame["order_purchase_timestamp"]
    features["estimated_delivery_lead_days"] = (
        frame["order_estimated_delivery_date"] - purchase
    ).dt.total_seconds() / 86400
    features["purchase_year"] = purchase.dt.year
    features["purchase_month"] = purchase.dt.month
    features["purchase_dayofweek"] = purchase.dt.dayofweek
    features["purchase_hour"] = purchase.dt.hour
    features["purchase_is_weekend"] = purchase.dt.dayofweek.ge(5).astype("int8")
    features["purchase_is_fixed_holiday"] = [
        int((month, day) in fixed_holiday_month_days)
        for month, day in zip(purchase.dt.month, purchase.dt.day)
    ]
    return features

feature_frames = {name: build_feature_frame(frame) for name, frame in split_frames.items()}
expected_columns = list(feature_frames["train"].columns)
for name in ["validation", "test"]:
    feature_frames[name] = feature_frames[name].reindex(columns=expected_columns)

forbidden = set(decisions["leakage_columns"] + decisions["identifier_or_high_cardinality_columns"])
leaked = forbidden.intersection(expected_columns)
assert not leaked, f"Leakage/high-cardinality columns reached the feature frame: {sorted(leaked)}"
print(f"Raw prediction-time features: {len(expected_columns)}")

Raw prediction-time features: 29


## Fit preprocessing on training only

In [3]:
categorical_features = decisions["base_categorical_features"]
numeric_features = [c for c in expected_columns if c not in categorical_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore", min_frequency=25, sparse_output=True
    )),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    sparse_threshold=1.0,
    verbose_feature_names_out=True,
)

X_train = sparse.csr_matrix(preprocessor.fit_transform(feature_frames["train"]))
X_validation = sparse.csr_matrix(preprocessor.transform(feature_frames["validation"]))
X_test = sparse.csr_matrix(preprocessor.transform(feature_frames["test"]))
feature_names = preprocessor.get_feature_names_out().tolist()

assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1] == len(feature_names)
assert np.isfinite(X_train.data).all()
assert np.isfinite(X_validation.data).all()
assert np.isfinite(X_test.data).all()
print(f"Transformed feature count: {len(feature_names)}")
print(f"Train matrix: {X_train.shape}, density={X_train.nnz / np.prod(X_train.shape):.3%}")

Transformed feature count: 142
Train matrix: (67529, 142), density=27.465%


## Save fitted transformers and final feature tables

In [4]:
matrices = {"train": X_train, "validation": X_validation, "test": X_test}
matrix_summary = {}
for name, matrix in matrices.items():
    sparse.save_npz(ARTIFACT_DIR / f"05_X_{name}.npz", matrix, compressed=True)
    y = split_frames[name]["late"].astype("int8").to_numpy()
    np.save(ARTIFACT_DIR / f"05_y_{name}.npy", y)
    split_frames[name][["order_id"]].to_csv(
        ARTIFACT_DIR / f"05_order_ids_{name}.csv.gz",
        index=False, compression="gzip",
    )
    matrix_summary[name] = {
        "rows": int(matrix.shape[0]),
        "columns": int(matrix.shape[1]),
        "nonzero": int(matrix.nnz),
        "late_labels": int(y.sum()),
    }

joblib.dump(preprocessor, ARTIFACT_DIR / "05_preprocessor.joblib")
(ARTIFACT_DIR / "05_feature_list.json").write_text(
    json.dumps(feature_names, indent=2), encoding="utf-8"
)
metadata = {
    "fit_split": "train only",
    "numeric_input_features": numeric_features,
    "categorical_input_features": categorical_features,
    "transformed_feature_count": len(feature_names),
    "matrix_format": "SciPy CSR sparse NPZ",
    "matrices": matrix_summary,
    "leakage_check_passed": True,
}
(ARTIFACT_DIR / "05_matrix_summary.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

reloaded = joblib.load(ARTIFACT_DIR / "05_preprocessor.joblib")
check_rows = sparse.csr_matrix(reloaded.transform(feature_frames["validation"].head(5)))
assert check_rows.shape == (5, len(feature_names))
print("Saved and reloaded the fitted preprocessor successfully.")

Saved and reloaded the fitted preprocessor successfully.
